# NFL Football Data Embeddings with Azure PostgreSQL & DiskANN

Welcome to the **NFL Football Data Embeddings** notebook!  
This notebook demonstrates how to generate, store, and index vector embeddings for NFL games, players, and plays using:

- **Azure PostgreSQL** for scalable, cloud-hosted relational storage
- **OpenAI Embeddings** for high-dimensional vector representations
- **DiskANN** for efficient approximate nearest neighbor search

---

## 🚀 Workflow Overview

1. **Database Setup:**  
    Connect to Azure PostgreSQL and enable vector search extensions.

2. **Embeddings Generation:**  
    - Generate embeddings for games, players, and plays using OpenAI models.
    - Store embeddings in dedicated tables for each entity.

3. **Indexing & Search:**  
    - Create DiskANN indexes for fast similarity search.
    - Enable semantic search and retrieval for advanced analytics.

---

## 📦 Use Cases

- **Semantic Search:** Find similar games, players, or plays based on context, not just keywords.
- **Recommendation Systems:** Suggest related plays or players using vector similarity.
- **Advanced Analytics:** Power downstream machine learning and AI applications.

---

> **Tip:**  
> Explore each section for code, explanations, and best practices for working with vector embeddings in modern data workflows.

---

Let's get started! 🏈

In [ ]:
# Add parent directory to sys.path to allow importing helpers and other modules
# This is required to use the embedding_service object, which is imported from helpers.embeddings_utils
import sys
import os
# Add the parent directory to sys.path so that helper modules can be imported
sys.path.append(os.path.abspath('..'))

In [ ]:
# Connect to the Azure PostgreSQL database and install required extensions for vector search
import psycopg2
from helpers.db_utils import get_football_connection_uri

# Get the connection URI for the football database
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

# Install the VECTOR extension for vector operations (if not already installed)
cur.execute("""
    CREATE EXTENSION IF NOT EXISTS 'VECTOR');
""")

# Install the PG_DISKANN extension for approximate nearest neighbor search (if not already installed)
cur.execute("""
    CREATE EXTENSION IF NOT EXISTS 'PG_DISKANN');
""")

conn.commit()
cur.close()
cur = conn.cursor()

## Games Embeddings

This section demonstrates how to generate and store vector embeddings for NFL games.  
We use OpenAI models to create high-dimensional representations of each game, capturing contextual and semantic information.

**Workflow:**
1. **Table Creation:**  
    Set up a dedicated table in Azure PostgreSQL to store game embeddings and metadata.
2. **Embedding Generation:**  
    For each game, generate an embedding using relevant game details (date, teams, week, etc.).
3. **Storage:**  
    Insert the embeddings and associated metadata into the database for efficient retrieval and analysis.
4. **Indexing:**  
    Create a DiskANN index on the embedding column to enable fast similarity search.

> These embeddings power semantic search, recommendations, and advanced analytics for NFL games.

In [ ]:
# Add parent directory to sys.path to allow importing helpers and other modules
# This is required to use the embedding_service object, which is imported from helpers.embeddings_utils
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
# Create the games_embeddings_diskann table to store game embeddings for vector search
import psycopg2
from helpers.db_utils import get_football_connection_uri
from helpers.embeddings_utils import embedding_service

# Connect to the Azure PostgreSQL database
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

# Drop the table if it exists and create a new one for storing game embeddings
cur.execute("""
    DROP TABLE IF EXISTS games_embeddings_diskann CASCADE;
    CREATE TABLE games_embeddings_diskann (
        vector_id SERIAL PRIMARY KEY,      -- Unique vector identifier
        gameid INTEGER NOT NULL,           -- Game ID
        gamedate DATE,                     -- Game date
        gametimeeastern TIME,              -- Game time (Eastern)
        hometeamabbr VARCHAR,              -- Home team abbreviation
        visitorteamabbr VARCHAR,           -- Visitor team abbreviation
        week INTEGER,                      -- Week number
        embedding_text VARCHAR,            -- Text used for embedding
        embedding_vector vector (1536) NOT NULL -- Embedding vector (1536 dimensions)
    );
""")
conn.commit()
cur.close()
cur = conn.cursor()

In [ ]:
# Generate embeddings for each game and store them in the games_embeddings_diskann table
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

# Fetch all games from the games table
cur.execute("SELECT * FROM games")
for row in cur.fetchall():
    gameId, gameDate, gametimeeastern, homeTeamAbbr, visitorTeamAbbr, week = row
    embedding_text = f"{gameId} {gameDate} {gametimeeastern} {homeTeamAbbr} {visitorTeamAbbr} {week}"
    response = await embedding_service.generate_embeddings([embedding_text])
    embedding_vector = response[0]
    cur.execute(
        "INSERT INTO games_embeddings_diskann (gameid, gamedate, gametimeeastern, hometeamabbr, visitorteamabbr, week, embedding_text, embedding_vector) VALUES (%s,%s,%s,%s,%s,%s,%s,%s)", 
        (gameId, gameDate,  gametimeeastern, homeTeamAbbr, visitorTeamAbbr, week, embedding_text, embedding_vector.tolist())
    )
print("All embeddings inserted into games_embeddings_diskann table.")
conn.commit()
cur.close()
conn.close()


In [ ]:
# Create index on the embedding vector column for efficient similarity search
query = """ CREATE INDEX games_embeddings_diskann_idx ON games_embeddings_diskann 
USING diskann (embedding_vector vector_cosine_ops)"""
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()
cur.execute(query)
conn.commit()
conn.close()

## Players Embeddings

In this section, we generate and store vector embeddings for NFL players using OpenAI models. These embeddings capture rich semantic information about each player, enabling advanced search and analytics.

**Workflow:**
1. **Table Creation:**  
    Set up a dedicated table in Azure PostgreSQL to store player embeddings and metadata.
2. **Embedding Generation:**  
    For each player, generate an embedding using relevant attributes (height, weight, college, position, etc.).
3. **Storage:**  
    Insert the embeddings and associated metadata into the database for efficient retrieval.
4. **Indexing:**  
    Create a DiskANN index on the embedding column to enable fast similarity search.

> These player embeddings power semantic search, recommendations, and player similarity analysis for NFL data applications.

In [ ]:
# Add parent directory to sys.path to allow importing helpers and other modules
# This is required to use the embedding_service object, which is imported from helpers.embeddings_utils
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import psycopg2
from helpers.db_utils import get_football_connection_uri
from helpers.embeddings_utils import embedding_service

# connect to Azure postgres database
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

# create product_desc table if it does not exist
cur.execute("""
    DROP TABLE IF EXISTS players_embeddings_diskann CASCADE;
    CREATE TABLE players_embeddings_diskann (
        vector_id SERIAL PRIMARY KEY,
        nflid INTEGER NOT NULL,
        height VARCHAR,
        weight INTEGER,
        birthdate DATE,
        collegename VARCHAR,
        position VARCHAR,
        displayname VARCHAR,
        embedding_text VARCHAR,
        embedding_vector vector (1536) NOT NULL 
    );
""")
conn.commit()
cur.close()

In [ ]:
import psycopg2
from helpers.db_utils import get_football_connection_uri
from helpers.embeddings_utils import embedding_service

# connect to Azure postgres database
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

#generate embeddings for product descriptions and store them in the product_desc table
cur.execute("SELECT * FROM players")
for row in cur.fetchall():
    nflId, height, weight, birthdate, collegename, position, displayname = row
    embedding_text = f"id:{nflId} ht:{height} wt:{weight} bday:{birthdate} coll/university:{collegename} pos:{position} name:{displayname}"
    response = await embedding_service.generate_embeddings([embedding_text])
    embedding_vector = response[0]
    cur.execute(
        "INSERT INTO players_embeddings_diskann (nflid, height, weight, birthdate, collegename, position, displayname, embedding_text, embedding_vector) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)", 
        (nflId, height, weight, birthdate, collegename, position, displayname, embedding_text, embedding_vector.tolist())
    )
print("All embeddings inserted into players_embeddings_diskann table.")
conn.commit()
cur.close()
conn.close()


In [ ]:
# Create index on the embedding vector column for efficient similarity search
query = """ CREATE INDEX players_embeddings_diskann_idx ON players_embeddings_diskann 
USING diskann (embedding_vector vector_cosine_ops)"""
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()
cur.execute(query)
conn.commit()
conn.close()

## Plays Embeddings

In this section, we generate and store vector embeddings for individual NFL plays using OpenAI models. These embeddings capture the rich context and semantics of each play, enabling advanced search, analytics, and recommendation scenarios.

**Workflow:**
1. **Table Creation:**  
    Set up a dedicated table in Azure PostgreSQL to store play embeddings and associated metadata.
2. **Embedding Generation:**  
    For each play, generate an embedding using detailed play attributes (description, quarter, down, formation, result, etc.).
3. **Storage:**  
    Insert the embeddings and metadata into the database for efficient retrieval and analysis.
4. **Indexing:**  
    Create a DiskANN index on the embedding column to enable fast similarity search.

> These play embeddings power semantic search, play similarity analysis, and intelligent recommendations for NFL data applications.

In [ ]:
# Add parent directory to sys.path to allow importing helpers and other modules
# This is required to use the embedding_service object, which is imported from helpers.embeddings_utils
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import psycopg2
from helpers.db_utils import get_football_connection_uri
from helpers.embeddings_utils import embedding_service

# connect to Azure postgres database
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

# create product_desc table if it does not exist
cur.execute("""
    DROP TABLE IF EXISTS plays_embeddings_diskann CASCADE;
    CREATE TABLE plays_embeddings_diskann (
        vector_id SERIAL PRIMARY KEY,
        gameid VARCHAR,
        playid BIGINT,
        playdescription TEXT,
        quarter INTEGER,
        down INTEGER,
        yardstogo INTEGER,
        possessionteam VARCHAR,
        playtype VARCHAR,
        yardlineside VARCHAR,
        yardlinenumber BIGINT,
        offenseformation VARCHAR,
        personnelo VARCHAR,
        defendersinthebox BIGINT,
        numberofpassrushers BIGINT,
        personneld VARCHAR,
        typedropback VARCHAR,
        presnapvisitorScore BIGINT,
        presnaphomescore BIGINT,
        gameclock TIME,
        absoluteyardlinenumber BIGINT,
        penaltycodes VARCHAR,
        penaltyjerseynumbers VARCHAR,
        passresult VARCHAR,
        offenseplayresult BIGINT,
        playresult BIGINT,
        epa FLOAT,
        isDefensivepi BOOLEAN,
        embedding_text VARCHAR,
        embedding_vector vector (1536) NOT NULL 
    );
""")
conn.commit()
cur.close()

In [ ]:
import psycopg2
from helpers.db_utils import get_football_connection_uri
from helpers.embeddings_utils import embedding_service

# connect to Azure postgres database
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

#generate embeddings for product descriptions and store them in the product_desc table
cur.execute("SELECT * FROM plays")
for num, row in enumerate(cur.fetchall()):
    (
        gameid, playid, playdescription, quarter, down, yardstogo, possessionteam, playtype,
        yardlineside, yardlinenumber, offenseformation, personnelo, defendersinthebox,
        numberofpassrushers, personneld, typedropback, presnapvisitorScore, presnaphomescore,
        gameclock, absoluteyardlinenumber, penaltycodes, penaltyjerseynumbers, passresult,
        offenseplayresult, playresult, epa, isDefensivepi
    ) = row
    embedding_text = (
        f"gameid:{gameid} playid:{playid} playdescription:{playdescription} quarter:{quarter} down:{down} "
        f"yardstogo:{yardstogo} possessionteam:{possessionteam} playtype:{playtype} yardlineside:{yardlineside} "
        f"yardlinenumber:{yardlinenumber} offenseformation:{offenseformation} personnelo:{personnelo} "
        f"defendersinthebox:{defendersinthebox} numberofpassrushers:{numberofpassrushers} personneld:{personneld} "
        f"typedropback:{typedropback} presnapvisitorScore:{presnapvisitorScore} presnaphomescore:{presnaphomescore} "
        f"gameclock:{gameclock} absoluteyardlinenumber:{absoluteyardlinenumber} penaltycodes:{penaltycodes} "
        f"penaltyjerseynumbers:{penaltyjerseynumbers} passresult:{passresult} offenseplayresult:{offenseplayresult} "
        f"playresult:{playresult} epa:{epa} isDefensivepi:{isDefensivepi}"
    )
    response = await embedding_service.generate_embeddings([embedding_text])
    embedding_vector = response[0]
    cur.execute(
        "INSERT INTO plays_embeddings_diskann (gameid, playid, playdescription, quarter, down, yardstogo, possessionteam, playtype, yardlineside, yardlinenumber, offenseformation, personnelo, defendersinthebox, numberofpassrushers, personneld, typedropback, presnapvisitorScore, presnaphomescore, gameclock, absoluteyardlinenumber, penaltycodes, penaltyjerseynumbers, passresult, offenseplayresult, playresult, epa, isDefensivepi, embedding_text, embedding_vector) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)", 
        (gameid, playid, playdescription, quarter, down, yardstogo, possessionteam, playtype, yardlineside, yardlinenumber, offenseformation, personnelo, defendersinthebox, numberofpassrushers, personneld, typedropback, presnapvisitorScore, presnaphomescore, gameclock, absoluteyardlinenumber, penaltycodes, penaltyjerseynumbers, passresult, offenseplayresult, playresult, epa, isDefensivepi, embedding_text, embedding_vector.tolist())
    )
    conn.commit()

print("All embeddings inserted into plays_embeddings_diskann table.")
conn.commit()
cur.close()
conn.close()


In [ ]:
# Create index on the embedding vector column for efficient similarity search
query = """ CREATE INDEX plays_embeddings_diskann_idx ON plays_embeddings_diskann 
USING diskann (embedding_vector vector_cosine_ops)"""
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()
cur.execute(query)
conn.commit()
conn.close()